# 🚀 SIRCCD - Entrenamiento YOLOv8 en Google Colab

Este notebook entrena un modelo YOLOv8 para detección de daños viales (baches y grietas) usando el dataset SIRCCD.

**GPU Recomendada**: T4 (15 GB VRAM)

**Tiempo estimado**: 6-8 horas (100 epochs)

**Dataset**: 57,976 imágenes (baches y grietas en vías públicas)

---

## 📋 Checklist Inicial

- [ ] Runtime > Change runtime type > GPU > T4

- [ ] Dataset subido a Google Drive (ver guía abajo)- [ ] Google Drive conectado

## 🔧 1. Setup Inicial

In [ ]:
# Verificar GPU disponible
!nvidia-smi

In [ ]:
# Instalar dependencias
!pip install -q ultralytics minio python-dotenv pillow

In [ ]:
# Montar Google Drive (para guardar modelo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Mantener sesión activa
from google.colab import output
output.enable_keepalive()

## 🗄️ 2. Descargar Dataset desde Google Drive

### Opción A: Dataset ya en Drive

Si ya subiste el dataset a Drive en la carpeta `SIRCCD_Dataset/`, salta a la siguiente celda.

### Opción B: Subir dataset a Drive

1. **Exportar desde MinIO local** (en tu PC):
```bash
# En tu terminal local
cd ml/datasets
python scripts/export_for_colab.py
```

2. **Subir a Google Drive**:
   - Crea carpeta `SIRCCD_Dataset` en tu Drive
   - Sube el archivo `sirccd_dataset_v1.0.0.zip` (aprox 15 GB)

### Opción C: Usar MinIO (avanzado)

In [ ]:
# Descargar y extraer dataset desde Google Drive
import os
import zipfile
from google.colab import drive

# Montar Drive (si no lo hiciste antes)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Ruta del dataset en Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/SIRCCD_Dataset/sirccd_dataset_v1.0.0.zip'

# Verificar que existe
if not os.path.exists(DRIVE_DATASET_PATH):
    print("❌ ERROR: Dataset no encontrado en Drive")
    print("   Sube el archivo a: MyDrive/SIRCCD_Dataset/sirccd_dataset_v1.0.0.zip")
    print("   O ejecuta: python ml/datasets/scripts/export_for_colab.py")
else:
    print("✅ Dataset encontrado en Drive")

In [ ]:
# Extraer dataset
import zipfile
from tqdm import tqdm

print("📦 Extrayendo dataset...")

# Extraer ZIP
with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zip_ref:
    # Obtener lista de archivos
    file_list = zip_ref.namelist()
    
    # Extraer con barra de progreso
    for file in tqdm(file_list, desc="Extrayendo"):
        zip_ref.extract(file, '/content/')

print("✅ Dataset extraído")

# Verificar estructura
import glob

train_images = len(glob.glob('/content/sirccd_dataset/images/train/*.jpg'))
val_images = len(glob.glob('/content/sirccd_dataset/images/val/*.jpg'))
test_images = len(glob.glob('/content/sirccd_dataset/images/test/*.jpg'))

print(f"
📊 Estructura del dataset:")
print(f"   Train: {train_images:,} imágenes")
print(f"   Val: {val_images:,} imágenes")
print(f"   Test: {test_images:,} imágenes")
print(f"   Total: {train_images + val_images + test_images:,} imágenes")

## 📝 3. Crear Archivo de Configuración YOLO

In [ ]:
# data.yaml para entrenamiento
data_yaml = """
# SIRCCD Dataset - Road Damage Detection
path: /content/sirccd_dataset
train: images/train
val: images/val
test: images/test

# Classes
names:
  0: bache
  1: grieta

# Number of classes
nc: 2

# Dataset info
# Total: 57,976 images
# Train: 40,543 (70%)
# Val: 11,614 (20%)
# Test: 5,819 (10%)
"""

!cat /content/sirccd_dataset/data.yaml

with open('/content/sirccd_dataset/data.yaml', 'w') as f:print("✅ data.yaml creado")

    f.write(data_yaml)

## 🎯 4. Entrenar Modelo YOLOv8

In [ ]:
from ultralytics import YOLO

# Cargar modelo pre-entrenado
model = YOLO('yolov8n.pt')  # nano - más rápido
# model = YOLO('yolov8s.pt')  # small - mejor precisión
# model = YOLO('yolov8m.pt')  # medium - requiere más VRAM

print("✅ Modelo cargado")

In [ ]:
# Configuración de entrenamiento
EPOCHS = 100
BATCH_SIZE = 16  # Ajusta según GPU (T4: 16-32, V100: 32-64)
IMG_SIZE = 640
PROJECT_NAME = 'sirccd-training'
RUN_NAME = 'baseline-yolov8n'

# Entrenar
results = model.train(
    data='/content/sirccd_dataset/data.yaml',
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,  # GPU
    amp=True,  # Mixed precision (2x más rápido)
    cache=True,  # Cache imágenes en RAM
    project=PROJECT_NAME,
    name=RUN_NAME,
    save_period=10,  # Guardar checkpoint cada 10 epochs
    patience=50,  # Early stopping si no mejora en 50 epochs
    verbose=True,
    
    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0
)

print("\n🎉 Entrenamiento completado!")

## 📊 5. Evaluar Modelo

In [ ]:
# Validar en dataset de validación
metrics = model.val()

print("\n📊 Métricas de Validación:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Visualizar predicciones de ejemplo
from IPython.display import Image, display
import glob

# Tomar algunas imágenes de ejemplo
test_images = glob.glob('/content/datasets/sirccd/images/*.jpg')[:5]

for img_path in test_images:
    results = model.predict(img_path, save=True, conf=0.25)
    
print("\n📸 Predicciones guardadas en:")
!ls -lh runs/detect/predict*/

# Mostrar una predicción
pred_images = glob.glob(f'{PROJECT_NAME}/{RUN_NAME}/val_batch*.jpg')
if pred_images:
    display(Image(filename=pred_images[0]))

## 💾 6. Guardar Modelo en Google Drive

In [ ]:
import shutil
from datetime import datetime

# Crear carpeta en Drive
drive_folder = '/content/drive/MyDrive/SIRCCD_Models'
os.makedirs(drive_folder, exist_ok=True)

# Timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_folder = f'{drive_folder}/{RUN_NAME}_{timestamp}'

# Copiar resultados completos
shutil.copytree(
    f'{PROJECT_NAME}/{RUN_NAME}',
    model_folder
)

print(f"✅ Modelo guardado en Google Drive:")
print(f"   {model_folder}")
print(f"\n📁 Archivos incluidos:")
print(f"   - weights/best.pt    (mejor modelo)")
print(f"   - weights/last.pt    (último epoch)")
print(f"   - results.csv        (métricas)")
print(f"   - confusion_matrix.png")
print(f"   - PR_curve.png")
print(f"   - F1_curve.png")

## 📤 7. (Opcional) Subir Modelo a MinIO

In [ ]:
# Subir best.pt a MinIO
import os

best_model_path = f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'
minio_object_name = f'models/{RUN_NAME}_{timestamp}/best.pt'

try:
    minio_client.fput_object(
        BUCKET_NAME,
        minio_object_name,
        best_model_path
    )
    print(f"✅ Modelo subido a MinIO: {minio_object_name}")
except Exception as e:
    print(f"❌ Error subiendo a MinIO: {e}")
    print("   (No te preocupes, el modelo está en Google Drive)")

## 🔔 8. Notificación de Finalización

In [ ]:
# Resumen final
import json

summary = {
    'model': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'mAP50': float(metrics.box.map50),
    'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'timestamp': timestamp,
    'drive_path': model_folder
}

print("\n" + "="*60)
print("🎉 ENTRENAMIENTO COMPLETADO")
print("="*60)
print(json.dumps(summary, indent=2))
print("="*60)

# Guardar resumen
with open(f'{model_folder}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Descarga los archivos desde Google Drive")
print("   O continúa experimentando en este notebook")

---

## 🚀 Próximos Pasos

1. **Descargar modelo desde Google Drive** a tu PC
2. **Evaluar en casos reales** con nuevas imágenes
3. **Fine-tuning**:
   - Probar YOLOv8s/m para mejor precisión
   - Ajustar data augmentation
   - Entrenar más epochs
4. **Optimizar para producción**:
   - Exportar a ONNX: `model.export(format='onnx')`
   - Exportar a TensorRT para Jetson Nano
5. **Integrar con backend** del proyecto SIRCCD

---

**Documentación**:
- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com)
- [Google Colab Tips](https://colab.research.google.com/notebooks/)

**Soporte**: Ver `ml/docs/CLOUD_TRAINING.md` en el repositorio